# Panorama (MultiDiffusion) → Modular Diffusers — E2E (training-free ultra-wide)

Ultra-wide FLUX beyond its native aspect (MultiDiffusion, arXiv:2302.08113). Publish PRIVATE
`remyxai/panorama-flux-modular` → load via `trust_remote_code` → a full **3072×1024** panorama →
**quantitative validation**: (a) no seam at the window boundaries — adjacent-column variance must show
no spike at a boundary x, and (b) no per-window repetition — adjacent windows must not be near-duplicates
(the failure of naive tiled generation). Also shows the naive-wide-gen control that motivates the method.
Upload `block.py` first. Runtime: A100 · `HUGGINGFACE_TOKEN` · accept FLUX.1-dev.

## 1 · Install + GPU + auth

In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf

In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16

## 2 · Publish PRIVATE (upload `block.py` first)

In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first."
open("modular_config.json","w").write(json.dumps({"_class_name":"PanoramaBlock","_diffusers_version":"0.41.0.dev0","auto_map":{"ModularPipelineBlocks":"block.PanoramaBlock"}},indent=2))
F="black-forest-labs/FLUX.1-dev"
def c(s,l,cl): return [None,None,{"pretrained_model_name_or_path":F,"revision":None,"subfolder":s,"type_hint":[l,cl],"variant":None}]
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name":"PanoramaBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
 "text_encoder":c("text_encoder","transformers","CLIPTextModel"),"tokenizer":c("tokenizer","transformers","CLIPTokenizer"),
 "text_encoder_2":c("text_encoder_2","transformers","T5EncoderModel"),"tokenizer_2":c("tokenizer_2","transformers","T5TokenizerFast"),
 "transformer":c("transformer","diffusers","FluxTransformer2DModel"),"vae":c("vae","diffusers","AutoencoderKL"),
 "scheduler":c("scheduler","diffusers","FlowMatchEulerDiscreteScheduler")},indent=2))
api=HfApi(); REPO="remyxai/panorama-flux-modular"; api.create_repo(REPO,private=True,repo_type="model",exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]: api.upload_file(path_or_fileobj=f,path_in_repo=f,repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))

## 3 · Load

In [ ]:
from diffusers import ModularPipeline
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/panorama-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__ == "PanoramaBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect PanoramaBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

## 4 · Spike first — no-op when one window covers the canvas

Run `smoke.ipynb` milestone B if you haven't: `window >= canvas` must reproduce **stock FLUX**
(bit-exact), which exercises exactly the per-window `img_ids` + scatter-average seam this notebook
relies on.

## 5 · The panorama (full settings)

3072×1024 — a 3:1 canvas at FLUX's native 1024 window with 50% overlap. The packed token grid is
64×192 and each window is 64×64 tokens, so the layout is 1 row × 5 columns: **5 window passes per
step** (≈5× a native 1024² step).

In [ ]:
import torch
from PIL import Image
from IPython.display import display

PROMPT = ("a sweeping panoramic view of a mountain range at golden hour, a winding river in the "
          "valley below, dramatic clouds, wide angle landscape photograph")
g = torch.Generator(DEV).manual_seed(0)
pan = pipe(prompt=PROMPT, height=1024, width=3072, window=1024, stride=512,
           num_inference_steps=28, guidance_scale=3.5, generator=g).images[0]
assert pan.size == (3072, 1024), pan.size
pan.save("panorama.png")
print("panorama:", pan.size)
display(pan.resize((1280, 427)))

## 6 · Quantitative — no seams, no repetition

The averaging is the quality lever, so measure both of the brief's failure modes directly on pixels:

**(a) Seam check.** Take the horizontal-gradient magnitude summed over rows, and compare the columns at
the **window boundaries** (x = 1024, 1536, 2048 for this layout) against the canvas median. A stitched
tiling shows a spike exactly there; a fused canvas does not.

**(b) Repetition check.** Compare the left and right **halves** of the panorama. Naive tiled/independent
generation repeats content per window (near-duplicate halves); a coherent panorama does not.

In [ ]:
import numpy as np
from PIL import Image

a = np.asarray(Image.open("panorama.png"), dtype=np.float32) / 255.0   # (H, W, 3)
H_, W_ = a.shape[:2]

# (a) seam check: column-wise total gradient energy vs the canvas median
grad = np.abs(np.diff(a, axis=1)).sum(axis=(0, 2))          # per column x
med = float(np.median(grad))
boundaries = [x for x in (1024, 1536, 2048) if 0 < x < W_]
ratios = {x: float(grad[x - 1:x + 2].max()) / med for x in boundaries}
print(f"gradient energy: canvas median = {med:.2f}")
for x, r in ratios.items():
    print(f"  boundary x={x}: peak/median = {r:.2f}")
seam_ok = all(r < 3.0 for r in ratios.values())
print("seam check:", "PASS (no boundary spike)" if seam_ok else "REVIEW (boundary spike -> raise the overlap)")

# (b) repetition check: are the two halves near-duplicates?
w = W_ // 2
corr = float(np.corrcoef(a[:, :w].reshape(-1), a[:, w:2 * w].reshape(-1))[0, 1])
print(f"\nhalf-vs-half correlation = {corr:.3f}  (near 1.0 = content repeated per window)")
rep_ok = corr < 0.95
print("repetition check:", "PASS (one coherent scene)" if rep_ok else "REVIEW (halves repeat -> content is tiling)")

print("\nE2E:", "PASS" if (seam_ok and rep_ok) else "REVIEW")

## 7 · Control — why window at all (the "woven blob")

Same prompt, same seed, denoised as **one naive wide latent** (no windows). FLUX's position ids and the
flow-match schedule are pushed far out of distribution, so this degrades into a smeared, textured blob.
This is the baseline MultiDiffusion replaces and the reason `mu` is computed at the window token count.

In [ ]:
import torch, numpy as np
from PIL import Image
from IPython.display import display

# naive wide generation: set window >= width so ONE window spans the whole 3072-wide canvas — a single
# pass over the full out-of-distribution sequence (3072/16 -> 192x64 = 12288 tokens vs native 4096).
g = torch.Generator(DEV).manual_seed(0)
naive = pipe(prompt=PROMPT, height=1024, width=3072, window=4096, stride=4096,
             num_inference_steps=28, guidance_scale=3.5, generator=g).images[0]
naive.save("naive_wide.png")

pan = np.asarray(Image.open("panorama.png"), dtype=np.float32) / 255.0
nai = np.asarray(naive, dtype=np.float32) / 255.0
print(f"mean |dx|  panorama (windowed, 5 passes) = {np.abs(np.diff(pan, axis=1)).mean():.4f}")
print(f"mean |dx|  naive wide (single window)   = {np.abs(np.diff(nai, axis=1)).mean():.4f}")
print("compare visually: the windowed panorama keeps a coherent scene; the naive one smears into a blob")
display(naive.resize((1280, 427)))

## Verdict

PASS = `loaded block: PanoramaBlock` + a 3072×1024 panorama + no boundary gradient spike (< 3× median)
+ halves not near-duplicates (< 0.95 correlation) + the single-window no-op spike in `smoke.ipynb`
matching stock FLUX. On pass: human confirms, flip the repo public, add the Colab badge + umbrella
collection.